# Nkwen Traders, Chapter 3: My Cleaning Notebook

**Goal:** turn the raw July-September 2026 sales file from all four branches into a clean, trustworthy dataset, and explain every decision you make.

**How this notebook is used.** You build this ONE notebook across Milestone 5:

| Section | Activity |
|---|---|
| 1. Audit (look, do not fix) | Why Data Is Never Clean |
| 2. Formatting: text, numbers, dates | Fixing Inconsistent Formatting |
| 3. Duplicates | Detecting & Handling Duplicates |
| 4. Missing values | Handling Missing Data |
| 5. Outliers and impossible values | Outlier Detection & Treatment |
| 6. Final checks, cleaning log and export | Project: Nkwen Traders - From Messy to Clean |

**What to submit each time:** the link to this notebook saved in your GitHub repository (File > Save a copy in GitHub). The reviewer reads the section that matches the activity, so keep the sections in this order and run all cells from top to bottom before you submit.

**Rules for the whole notebook**
- Never change `raw`. It is your "before" picture. All cleaning happens on `df`.
- Do not use `inplace=True`. Always assign the result back: `df["col"] = df["col"].something()`.
- Do not reset the index until Section 6. The self-checks use it to compare `df` with `raw`.
- After every change, check it worked (counts before and after).

In [ ]:
# Setup: run this cell first
import pandas as pd
import numpy as np

URL = "https://innovakickstarter.com/course-files/data-science/data/nkwen_traders_ch3_sales_raw.csv"
raw = pd.read_csv(URL)   # the untouched "before" data. Never change it.
df = raw.copy()          # your working copy. All cleaning happens here.

print("pandas version:", pd.__version__)
print("Rows and columns:", raw.shape)
raw.head()

---
## Section 1. Audit: look, do not fix
*Activity: Why Data Is Never Clean*

In this section you only **look**. Do not change `df` yet. Write down every problem you see. For each one, think about the real-world cause: who typed it, which device, which branch?

In [ ]:
# Step 1.1 - Overview: column types and missing values
# TODO: show raw.info()
# TODO: show the number of missing values in each column: raw.isna().sum()

In [ ]:
# Step 1.2 - Text columns: how many spellings does each value have?
# TODO: for each of these columns, print value_counts(dropna=False)
text_cols = ["Branch", "Category", "PaymentMethod", "CustomerType", "SalesRep", "Notes"]
for col in text_cols:
    pass  # replace pass with your code
# Tip: wrap a value in repr() to see hidden spaces, for example print(repr(value))

In [ ]:
# Step 1.3 - The "number" and date columns
# TODO: show 15 random rows of these columns: raw[cols].sample(15, random_state=1)
cols = ["Date", "Time", "Quantity", "UnitPrice_FCFA", "TotalSale_FCFA"]

# TODO: how many prices can pandas NOT read as numbers, compared with the real blanks?
# pd.to_numeric(raw["UnitPrice_FCFA"], errors="coerce").isna().sum()   versus   raw["UnitPrice_FCFA"].isna().sum()
# Do the same for Quantity and TotalSale_FCFA.

In [ ]:
# Step 1.4 - Duplicates and notes
# TODO: count rows that are exact copies of another row: raw.duplicated().sum()
# TODO: count TransactionIDs that appear more than once: raw.duplicated(subset="TransactionID").sum()
# TODO: show every row where Notes is not empty: raw[raw["Notes"].notna()]

### My audit table
Fill in at least 8 rows. "Rows affected" can be an estimate, but show the code output that supports it.

| # | Issue | Column(s) | Rows affected | One real example | Likely real-world cause | First idea for the fix |
|---|---|---|---|---|---|---|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |
| 4 | | | | | | |
| 5 | | | | | | |
| 6 | | | | | | |
| 7 | | | | | | |
| 8 | | | | | | |

**The issue that worries me most for the business, and why (2-3 sentences):**

*(write here)*

---
## Section 2. Formatting: text, numbers and dates
*Activity: Fixing Inconsistent Formatting*

Order of work: branch names, then categories, then the three number columns, then dates. After each step, check the result.

In [ ]:
# Step 2.1 - Branch names
# Plan: remove outer spaces -> squeeze double spaces -> Title Case -> map the spellings that are still wrong.
# TODO 1: df["Branch"] = df["Branch"].str.strip() ... add .str.replace(r"\s+", " ", regex=True) and .str.title()
# TODO 2: print df["Branch"].value_counts() and fill this dictionary with the spellings that are still wrong
branch_fixes = {
    # "wrong spelling": "correct name",
}
# TODO 3: df["Branch"] = df["Branch"].replace(branch_fixes)
# TODO 4: print df["Branch"].value_counts() again

In [ ]:
# Step 2.2 - Categories: same method as branches
category_fixes = {
    # "wrong spelling": "correct name",
}
# TODO: clean df["Category"] (strip, Title Case, then .replace(category_fixes)) and print value_counts()

### Step 2.3 - Numbers stored as text
Values such as `1 500 XAF`, `1,500 XAF`, `XAF 1500` and `12 pcs` are text, so pandas cannot add them up.
`pd.to_numeric(col, errors="coerce")` on its own would **silently turn them into NaN** (you measured this in Step 1.3).
So first remove every character that is not a digit, then convert.

**Keep the minus sign.** A few rows have negative numbers. If your pattern deletes `-`, a refund becomes a sale.

In [ ]:
def to_whole_number(series):
    """Turn text like '1 500 XAF', '1,500 XAF', 'XAF 1500' or '12 pcs' into numbers."""
    # TODO 1: text = series.astype("string")
    # TODO 2: keep only digits and the minus sign: text.str.replace(r"[^0-9-]", "", regex=True)
    # TODO 3: return pd.to_numeric(..., errors="coerce")
    return series  # replace this line

missing_before = {}
missing_after = {}
for col in ["Quantity", "UnitPrice_FCFA", "TotalSale_FCFA"]:
    missing_before[col] = df[col].isna().sum()
    df[col] = to_whole_number(df[col])
    missing_after[col] = df[col].isna().sum()
    print(f"{col}: missing before {missing_before[col]}, after {missing_after[col]}, dtype {df[col].dtype}")

### Step 2.4 - Dates: the day-first trap
In Cameroon, `05/07/2026` means **5 July**. Many tools read it month-first, as **7 May**. No error appears: the date is simply wrong.
Run the demo below and compare the two results.

Safest method: parse **each format explicitly** with `format=...` and `errors="coerce"`, then combine the results with `.fillna()`.
Avoid `format="mixed"`: it guesses each value separately. Its guesses changed between pandas versions, and with `dayfirst=True` recent versions can even read ISO dates such as `2026-07-01` as 7 January.

In [ ]:
# Demo: run this cell and read the output
example = pd.Series(["05/07/2026", "25/07/2026"])
print(pd.to_datetime(example, format="%m/%d/%Y", errors="coerce"))  # month-first: 7 May, and NaT for the 25th
print(pd.to_datetime(example, format="%d/%m/%Y", errors="coerce"))  # day-first: 5 July and 25 July (correct)

In [ ]:
# TODO: look at raw["Date"].sample(20) again and list the date formats you found.
# For each format, parse with an explicit format string. Unmatched values become NaT.
# Format codes: %Y = 2026, %m = 07, %d = 05, %b = Jul
# parsed_a = pd.to_datetime(df["Date"], format="...", errors="coerce")
# parsed_b = pd.to_datetime(df["Date"], format="...", errors="coerce")
# parsed_c = ...
# TODO: combine them, for example: df["Date"] = parsed_a.fillna(parsed_b).fillna(parsed_c)
# TODO: print df["Date"].isna().sum(), df["Date"].min() and df["Date"].max()

In [ ]:
# Self-check: formatting (run this after Steps 2.1-2.4)
expected_branches = {"Nkwen", "Mile 4", "Commercial Avenue", "Bafoussam Centre"}
assert set(df["Branch"].unique()) == expected_branches, f"Branch still has other spellings: {set(df['Branch'].unique()) - expected_branches}"
assert df["Category"].nunique() == 7, "Nkwen Traders sells 7 categories. Look at df['Category'].value_counts() again."
for col in ["Quantity", "UnitPrice_FCFA", "TotalSale_FCFA"]:
    assert pd.api.types.is_numeric_dtype(df[col]), f"{col} is not numeric yet."
    assert missing_after[col] == missing_before[col], f"Converting {col} created new missing values. Some text was not cleaned before pd.to_numeric."
assert (df["Quantity"] < 0).any(), "Negative quantities disappeared. Does your pattern keep the minus sign?"
assert pd.api.types.is_datetime64_any_dtype(df["Date"]), "Date is not a datetime column yet."
assert df["Date"].notna().all(), "Some dates could not be parsed (NaT). Did you cover every format?"
# Every parsed date must give back its original text in one of the formats you found:
original = raw.loc[df.index, "Date"]
same = pd.Series(False, index=df.index)
for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%b %d %Y"]:
    same = same | (df["Date"].dt.strftime(fmt) == original)
assert same.all(), "Some dates changed meaning (day and month swapped?). Use explicit formats and day-first for DD/MM/YYYY."
print("Section 2 self-check passed: branches, categories, numbers and dates look right.")

**My notes on formatting (3-5 sentences):** which date method did you use and why? What would have gone wrong with month-first parsing or with `pd.to_numeric` alone?

*(write here)*

---
## Section 3. Duplicates
*Activity: Detecting & Handling Duplicates*

Two kinds of copy can hide in sales data: **exact copies** (every column identical) and **sync copies** (same TransactionID, but the Time is a few seconds later because a phone re-sent the sale). Rows with **different** TransactionIDs are different sales, even if they look alike.

In [ ]:
# Step 3.1 - Count both kinds
# TODO: exact copies: df.duplicated().sum()
# TODO: repeated TransactionIDs: df.duplicated(subset="TransactionID").sum()

In [ ]:
# Step 3.2 - Look at the repeated IDs side by side
# keep=False marks ALL copies, so you can compare them
# TODO: dups = df[df.duplicated(subset="TransactionID", keep=False)].sort_values(["TransactionID", "Time"])
# TODO: show dups.head(12). Which columns differ between the copies?

In [ ]:
# Step 3.3 - Lookalikes: same day, branch, product and quantity, but DIFFERENT TransactionIDs
# TODO: lookalikes = df[df.duplicated(subset=["Date", "Branch", "Product", "Quantity"], keep=False)
#                       & ~df.duplicated(subset="TransactionID", keep=False)]
# TODO: show a few. Are these the same sale or two customers buying the same thing? Look at Time and TotalSale_FCFA.

In [ ]:
# Step 3.4 - Remove the true duplicates (keep the first copy)
rows_before = len(df)
# TODO: df = df.drop_duplicates(subset="TransactionID", keep="first")
rows_after = len(df)
print("Removed", rows_before - rows_after, "rows")

In [ ]:
# Self-check: duplicates
assert df["TransactionID"].is_unique, "Some TransactionIDs still appear more than once."
assert rows_before - rows_after >= raw.duplicated().sum(), "You removed fewer rows than there are exact copies. Did you use subset='TransactionID'?"
assert df["TransactionID"].nunique() == raw["TransactionID"].nunique(), "You lost whole sales. Remove copies only, never every row of a sale."
print("Section 3 self-check passed: every sale appears once and no sale was lost.")

**My duplicate rule (3-5 sentences):** what counts as a duplicate in this data, how you told sync copies from real repeat purchases, and what you would ask the IT person who runs the phone app.

*(write here)*

---
## Section 4. Missing values
*Activity: Handling Missing Data*

There is no single right answer for missing data. The right choice depends on **which column** is missing, **why** it is missing, and whether you can **recover** it from other columns.
Options: recover it, fill it with a label such as "Unknown", fill it with a number (median, forward fill), drop the row, or keep it aside and ask the owner.

In [ ]:
# Step 4.1 - What is missing now (after formatting and duplicates)?
# TODO: df.isna().sum()

In [ ]:
# Step 4.2 - Recover what can be recovered
# If Quantity and TotalSale_FCFA are present, the missing UnitPrice_FCFA is TotalSale_FCFA / Quantity.
price_missing = df["UnitPrice_FCFA"].isna() & df["Quantity"].notna() & df["TotalSale_FCFA"].notna()
print("Prices you can recover:", price_missing.sum())
# TODO: df.loc[price_missing, "UnitPrice_FCFA"] = df.loc[price_missing, "TotalSale_FCFA"] / df.loc[price_missing, "Quantity"]
# TODO: check that the recovered prices are whole numbers and look sensible for their products

In [ ]:
# Step 4.3 - Rows you cannot recover
# If BOTH Quantity and TotalSale_FCFA are missing, you do not know how much was sold.
cannot_recover = df["Quantity"].isna() & df["TotalSale_FCFA"].isna()
print("Rows you cannot recover:", cannot_recover.sum())
# TODO: keep them aside to show the branch manager: to_ask = df[cannot_recover].copy()
# TODO: remove them from the sales table:        df = df[~cannot_recover].copy()

In [ ]:
# Step 4.4 - Non-critical text columns: keep the sale, label the gap
# Correct pandas 2/3 style: assign the result back. Do NOT write df["col"].fillna(..., inplace=True).
# TODO: for PaymentMethod, SalesRep and CustomerType: df[col] = df[col].fillna("Unknown")

In [ ]:
# Demo: when ARE ffill() and bfill() right? Run this cell.
# A stock count taken some mornings only: the level stays the same until the next count.
stock = pd.Series([40, None, None, 35, None],
                  index=pd.date_range("2026-07-01", periods=5, freq="D"), name="RiceBags_in_stock")
print(stock.ffill())   # carry the last known count forward
print(stock.bfill())   # use the next known count
# Question for your notes: why would df["SalesRep"].ffill() be WRONG for our sales table?

In [ ]:
# Self-check: missing values
key_cols = ["Quantity", "UnitPrice_FCFA", "TotalSale_FCFA", "PaymentMethod", "SalesRep", "CustomerType"]
assert df[key_cols].isna().sum().sum() == 0, f"Still missing: {df[key_cols].isna().sum()[lambda s: s > 0].to_dict()}"
assert len(to_ask) > 0, "Keep the unrecoverable rows in `to_ask` so you can show them to the manager."
assert (df["UnitPrice_FCFA"] % 1 == 0).all(), "Some prices are not whole FCFA. Check your recovery step."
print("Section 4 self-check passed: no gaps left in key columns, and the unrecoverable rows are kept aside.")

**My missing-data decisions** (one line per column: what was missing, what you did, why). Include one sentence on why filling a missing price with the average price of all products would be wrong here.

| Column | What was missing | My decision | Why |
|---|---|---|---|
| | | | |

---
## Section 5. Outliers and impossible values
*Activity: Outlier Detection & Treatment*

An outlier is a value far from the others. It can be an **error** (fix it or remove it) or a **real event** (keep it). Your job is to investigate each one, not to delete everything unusual.

In [ ]:
# Step 5.1 - First look
# TODO: df[["Quantity", "UnitPrice_FCFA", "TotalSale_FCFA"]].describe()
# Which minimum and maximum values look impossible or surprising?

In [ ]:
# Step 5.2 - The naive IQR rule on the whole TotalSale_FCFA column
# TODO: q1 = df["TotalSale_FCFA"].quantile(0.25); q3 = ...; iqr = q3 - q1
# TODO: flagged = df[df["TotalSale_FCFA"] > q3 + 1.5 * iqr]
# TODO: print(len(flagged)) and flagged["Product"].value_counts().head()
# Question: are these sales errors, or simply expensive products?

In [ ]:
# Optional picture: one box per category (pandas draws it with matplotlib)
# df.boxplot(column="TotalSale_FCFA", by="Category", rot=45, figsize=(9, 4))

In [ ]:
# Step 5.3 - Judge each value against its OWN product
# Quantity z-score inside each product: how unusual is this quantity for this product?
by_product = df.groupby("Product")["Quantity"]
qty_z = (df["Quantity"] - by_product.transform("mean")) / by_product.transform("std")
# TODO: show the rows with qty_z.abs() > 3 (columns TransactionID, Product, Quantity, UnitPrice_FCFA, TotalSale_FCFA, CustomerType, Notes)

# Price compared with the product's median price
price_ratio = df["UnitPrice_FCFA"] / df.groupby("Product")["UnitPrice_FCFA"].transform("median")
# TODO: show the rows where price_ratio > 3 or price_ratio < 1/3

In [ ]:
# Step 5.4 - The consistency check: TotalSale_FCFA should equal Quantity x UnitPrice_FCFA
mismatch = df["Quantity"] * df["UnitPrice_FCFA"] != df["TotalSale_FCFA"]
# TODO: show df[mismatch]
# For each row, compare TotalSale_FCFA / UnitPrice_FCFA with Quantity, and TotalSale_FCFA / Quantity with UnitPrice_FCFA.
# Which column was mistyped? By how much?

In [ ]:
# Step 5.5 - Read the Notes column for every unusual row
# TODO: show df[df["Notes"].notna()]
# Which rows are real events to KEEP? Which rows are not sales at all?

In [ ]:
# Step 5.6 - Apply your decisions
# Fix typos with df.loc[...] = ... ; move refunds into their own table; keep real events.
# TODO: fix each mistyped value (use the TransactionID or a rule based on the consistency check)
# TODO: refunds = df[df["Quantity"] < 0].copy()
# TODO: df = df[df["Quantity"] > 0].copy()

In [ ]:
# Self-check: outliers
assert (df["Quantity"] * df["UnitPrice_FCFA"] == df["TotalSale_FCFA"]).all(), "Some rows still fail TotalSale = Quantity x UnitPrice."
assert (df["Quantity"] > 0).all(), "The sales table still has zero or negative quantities. Refunds belong in their own table."
assert len(refunds) > 0, "Keep the refunds in a separate `refunds` table: the manager needs to see them."
assert df["Notes"].str.contains("order", case=False, na=False).any(), "You removed the genuine bulk orders (rows with Notes). Real big sales must stay."
print("Section 5 self-check passed: totals are consistent, refunds are separate, real big orders are kept.")

### My outlier decisions
One row per unusual value you investigated.

| TransactionID | What looks wrong | Evidence (columns, Notes, product median) | Decision: fix / remove / keep / separate | Why |
|---|---|---|---|---|
| | | | | |

**Why the naive IQR rule was not enough here (2-3 sentences):**

*(write here)*

---
## Section 6. Final checks, cleaning log and export
*Activity: Project: Nkwen Traders - From Messy to Clean*

Polish this notebook so a manager or another analyst can trust it: final types, a before/after comparison, a cleaning log, questions for the manager, and the exported files.

In [ ]:
# Step 6.1 - Final types and tidy index
# TODO: convert Quantity, UnitPrice_FCFA and TotalSale_FCFA to whole numbers: df[col] = df[col].astype("int64")
# TODO: df = df.sort_values(["Date", "Time"]).reset_index(drop=True)

In [ ]:
# Step 6.2 - Before vs after
# TODO: build a small table comparing raw and df, for example:
# comparison = pd.DataFrame({
#     "Before (raw)": [len(raw), raw.isna().sum().sum(), raw["Branch"].nunique(), raw["Category"].nunique(), raw["TransactionID"].duplicated().sum()],
#     "After (df)":   [len(df),  df.drop(columns="Notes").isna().sum().sum(), df["Branch"].nunique(), df["Category"].nunique(), df["TransactionID"].duplicated().sum()],
# }, index=["Rows", "Missing cells (excluding Notes)", "Branch spellings", "Category spellings", "Repeated IDs"])
# Note: in "before", count missing cells WITHOUT Notes too, so you compare like with like.
# TODO: also show raw.dtypes next to df.dtypes

In [ ]:
# Self-check: final dataset
assert df["TransactionID"].is_unique, "Repeated TransactionIDs."
assert df.drop(columns="Notes").isna().sum().sum() == 0, "Missing values remain outside Notes."
for col in ["Quantity", "UnitPrice_FCFA", "TotalSale_FCFA"]:
    assert df[col].dtype == "int64", f"{col} should be int64 (whole FCFA)."
assert df["Date"].between("2026-07-01", "2026-09-30").all(), "Dates outside July-September 2026."
assert (df["Quantity"] * df["UnitPrice_FCFA"] == df["TotalSale_FCFA"]).all(), "TotalSale must equal Quantity x UnitPrice."
assert (df["Quantity"] > 0).all(), "Only sales belong in the sales table."
assert len(df) + len(to_ask) + len(refunds) + raw.duplicated(subset="TransactionID").sum() == len(raw), \
    "Rows do not add up: clean sales + rows to ask about + refunds + removed copies should equal the raw row count."
print("Final self-check passed. Every raw row is accounted for.")

### Cleaning log
| Step | Issue | Rows affected | Action | Why |
|---|---|---|---|---|
| 1 | | | | |

### Questions for the manager (stakeholder memo)
Write 3-6 short questions or flags for Nkwen Traders management: things you could not decide alone, or assumptions they should confirm. For each, say what you did for now.

*(write here)*

In [ ]:
# Step 6.3 - Export
# TODO: df.to_csv("nkwen_ch3_clean.csv", index=False)
# TODO: refunds.to_csv("nkwen_ch3_refunds.csv", index=False)
# TODO: to_ask.to_csv("nkwen_ch3_rows_to_check.csv", index=False)
# In Colab, download the files (left sidebar > Files, or the code below) and upload them to your GitHub repository.
# from google.colab import files
# files.download("nkwen_ch3_clean.csv")

---
## Before you submit
- [ ] Runtime > Restart session and run all: every cell runs from top to bottom without errors.
- [ ] All the self-checks for the sections you have finished print "passed".
- [ ] The markdown notes for the current section are filled in, in your own words.
- [ ] The notebook is saved in your GitHub repository (File > Save a copy in GitHub) and the link opens for someone who is not logged in.
- [ ] For the project: `nkwen_ch3_clean.csv`, `nkwen_ch3_refunds.csv` and `nkwen_ch3_rows_to_check.csv` are uploaded to the same repository, with a README.